# In Silico Protein Digestion using Rapid Peptides Generator

Rapid Peptides Generator (RPG), is a standalone software dedicated to predict proteases-induced cleavage sites on sequences.

RPG is a python tool taking a (multi-)fasta/fastq file of proteins as input and digest each of them. We will use this tool to digest the full proteome using trypsin. We will adjust the cleavage rules to match the ones in MaxQuant. This either means Trypsin cleaves after every L and K or it cleaves after every L and K except if it is followed by P.

The resulting peptides contain information about positions of cleavage site, peptide sequences, length, mass as well as an estimation of isoelectric point (pI) of each peptide. Results are outputted in multi-fasta, CSV or TSV file.

# Setup
## Import and install Packages

In [ ]:
import sys
import os
import session_info

# Add the '0_functions' folder to sys.path
sys.path.append(os.path.join(os.getcwd(), '..', '00_functions'))

In [ ]:
import pandas as pd
import re
from functions import read_fastafile

In [ ]:
# Display session information
session_info.show()

In [ ]:
!pip install rpg

In [ ]:
!rpg --help

## Set working directory

In [ ]:
cwd = os.getcwd()
if not cwd.endswith("Isoform_Database/Jupyter_environment/Isoform_Database_SIAF/02_Isoform_Databse"):
    print("WARNING: The working directory is not set to the '02_02_Isoform_Databse'!")
    print("Current working directory:", cwd)
else:
    print(f"Working directory is correctly set to the '02_02_Isoform_Databse' folder (\"{cwd}\").")

In [ ]:
# Data directories for fasta data
input_dir = "data/digestion_input"
output_dir = "data/digestion_output"
fasta_dir = "../01_UniProt/data/raw_data/fasta"

## Read in Fasta files

In [ ]:
fasta_full_proteome = os.path.join(fasta_dir, 'uniprotkb_full_proteome_2026_03_31.fasta')

# Performing digestion with RPG
I have defined two new enzymes. MaxQuant_Trypsin cleaves after every R and K and MaxQuant_Trypsin_P cleaves after every R and K except if they are followed by P.

In [ ]:
#Show list of possible Enzymes to check which number is Trypsin. The RPG definition is number 42, my new MaxQuant definitions are 47 and 48.
!rpg -l

Trypsin cleavage rules in RPG:
Trypsin preferentially cleaves after K or R (P1). It will not cleave after K followed by P in P1’ except if W in P2. It will not cleave after R followed by P in P1’ except if M in P2. It will not cleave CKD, DKD, CKH, CKY, CRK, RRH nor RRR.

For the digestion of the entire proteome I used Euler. The script that I used is shown below. The fragments are filtered so that we only keep the ones that are 7 amino acids long or longer.

In [ ]:
digestion_full_proteome_Trypsin = read_fastafile(os.path.join(output_dir, 'digestion_full_proteome_Trypsin.fasta'))
digestion_full_proteome_Trypsin_P = read_fastafile(os.path.join(output_dir, 'digestion_full_proteome_Trypsin_P.fasta'))

In [ ]:
digestion_Trypsin_filtered = digestion_full_proteome_Trypsin[digestion_full_proteome_Trypsin["len"] >= 7]
digestion_Trypsin_filtered = digestion_Trypsin_filtered[digestion_Trypsin_filtered["len"] <= 52]

digestion_TrypsinP_filtered = digestion_full_proteome_Trypsin_P[digestion_full_proteome_Trypsin_P["len"] >= 7]
digestion_TrypsinP_filtered = digestion_TrypsinP_filtered[digestion_TrypsinP_filtered["len"] <= 52]

In [ ]:
digestion_Trypsin_filtered

In [ ]:
#save filtered df as csv
digestion_Trypsin_filtered.to_csv(os.path.join(output_dir, 'digestion_Trypsin_filtered.csv'), index=False)
digestion_TrypsinP_filtered.to_csv(os.path.join(output_dir, 'digestion_TrypsinP_filtered.csv'), index=False)

In [ ]:
digestion_Trypsin_filtered["ID"].nunique()